# 01 — Deep Blue data: loading, timing and sanity checks

This notebook loads the University of Michigan Deep Blue Lu-177 DOTATATE datasets (patients 4 and 6), which are also the reference cases of the SNMMI Lu-177 Dosimetry Challenge.

For each patient there are four post-therapy SPECT/CT acquisitions, organ and lesion masks registered to each SPECT, and a time-integrated activity (TIA) map computed with MIM.

| Patient | Imaging days after injection |
|---|---|
| 4 | 0, 1, 4, 5 |
| 6 | 0, 1, 4, 8 |

**Goals**
1. Read every DICOM series into a 3D volume with correct geometry.
2. Extract the acquisition times relative to injection.
3. Convert the SPECT images from decay-corrected to physical activity concentration.
4. Check the whole-field-of-view activity and the TIA map for consistency.
5. Preview organ and lesion time-activity curves.
6. Save everything as NIfTI plus a JSON file with the timing, for the next notebooks.

**About this notebook.** Deep Blue SPECT/CT DICOM to NIfTI with per-patient decay-correction handling.
- **Inputs:** `$LU177_DATASET/Deep Blue Dataset/` (DICOM, MIM TIA map, VOI masks).
- **Outputs:** `data/processed/deepblue/patient_{4,6}/` (NIfTI per scan, `timing.json`).
- **Run time:** minutes.
- **Environment:** .venv.

## Decay correction

The SPECT headers report `Units = BQML` and `DECY` in `CorrectedImage`, but the reference time of the decay correction is **not the same for the two patients**:

| Patient | `DecayCorrection` | Meaning |
|---|---|---|
| 4 | `ADMIN` | decay-corrected to the administration time |
| 6 | `START` | decay-corrected to the start of the acquisition, i.e. already physical activity |

For dosimetry we need the activity physically present at the acquisition time. For `ADMIN` images:

$$A_\text{phys}(t) = A_\text{DC}(t)\, e^{-\lambda_\text{phys} t}, \qquad \lambda_\text{phys} = \frac{\ln 2}{T_{1/2}}$$

with $t$ the time from injection to the start of the acquisition and $T_{1/2}$ read from the header (`RadionuclideHalfLife`). `START` images are used as they are. The convention is therefore read from every scan and never assumed.

For comparison, a decay-corrected-to-injection version is also kept for every scan ($A_\text{DC} = A_\text{phys}\, e^{\lambda_\text{phys} t}$), so that both patients are handled the same way downstream.

The two conventions were confirmed against the MIM TIA map (see the organ-level check below): applying the `ADMIN` correction to patient 6 underestimates the kidney TIA by about 30%.

In [ ]:
import json
from datetime import datetime
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import pydicom
import SimpleITK as sitk

WS = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "src" / "paths.py").exists())
sys.path.insert(0, str(WS / "src"))
from paths import DATASET
DATA = DATASET / "Deep Blue Dataset"
OUT = WS / "data" / "processed" / "deepblue"
PATIENTS = ["patient_4", "patient_6"]
OUT.mkdir(parents=True, exist_ok=True)

## DICOM helpers

The series are stored slice by slice. Each slice can carry its own `RescaleSlope` (the TIA map does), so the volume is assembled manually: slices are sorted along the slice normal and each one is rescaled with its own slope and intercept before stacking.

Some folders of patient 6 also contain a non-image object (a Real World Value Mapping file, "PET SUV Factors"), so only files with pixel data are kept. Filtering on `Modality` would be wrong: one valid slice of patient 6 scan 2 is tagged `NM` instead of `PT`.

In [ ]:
def parse_dt(date, time):
    s = f"{date}{time}".split(".")[0]
    return datetime.strptime(s, "%Y%m%d%H%M%S")


def read_series(folder):
    slices = [pydicom.dcmread(f) for f in sorted(Path(folder).glob("*.dcm"))]
    slices = [d for d in slices if "PixelData" in d]
    first = slices[0]
    row = np.array(first.ImageOrientationPatient[:3], dtype=float)
    col = np.array(first.ImageOrientationPatient[3:], dtype=float)
    normal = np.cross(row, col)
    slices.sort(key=lambda d: float(np.dot(normal, np.array(d.ImagePositionPatient, dtype=float))))
    positions = np.array([np.dot(normal, np.array(d.ImagePositionPatient, dtype=float)) for d in slices])
    if len(slices) > 2 and not np.allclose(np.diff(positions), np.diff(positions)[0], atol=1e-2):
        raise ValueError(f"non-uniform slice spacing in {folder}")
    volume = np.stack([
        d.pixel_array.astype(np.float64) * float(d.get("RescaleSlope", 1.0)) + float(d.get("RescaleIntercept", 0.0))
        for d in slices
    ]).astype(np.float32)
    dz = float(np.median(np.diff(positions))) if len(slices) > 1 else float(first.get("SliceThickness", 1.0))
    ps = [float(v) for v in first.PixelSpacing]
    image = sitk.GetImageFromArray(volume)
    image.SetSpacing((ps[1], ps[0], dz))
    image.SetOrigin(tuple(float(v) for v in slices[0].ImagePositionPatient))
    image.SetDirection(tuple(np.column_stack([row, col, normal]).ravel()))
    return image, slices[0]


def header(folder):
    for f in sorted(Path(folder).glob("*.dcm")):
        h = pydicom.dcmread(f, stop_before_pixels=True)
        if "ImagePositionPatient" in h:
            return h
    raise FileNotFoundError(f"no image slice in {folder}")


def voxel_ml(image):
    return float(np.prod(image.GetSpacing())) / 1000.0


def same_grid(a, b, tol=1e-3):
    return (a.GetSize() == b.GetSize()
            and np.allclose(a.GetSpacing(), b.GetSpacing(), atol=tol)
            and np.allclose(a.GetOrigin(), b.GetOrigin(), atol=tol)
            and np.allclose(a.GetDirection(), b.GetDirection(), atol=tol))


def to_grid(image, reference, nearest=False, default=0.0):
    interp = sitk.sitkNearestNeighbor if nearest else sitk.sitkLinear
    return sitk.Resample(image, reference, sitk.Transform(), interp, default, image.GetPixelID())

## Acquisition inventory

One row per SPECT acquisition. `t_h` is the time from injection to the start of the acquisition. Check that the injected activity is close to the nominal 7.4 GBq, that the times match the table above and that units are the same for every scan. The `decay_correction` column decides how each scan is converted to physical activity: it must be constant within a patient, and only `ADMIN` and `START` are accepted.

In [ ]:
rows = []
for patient in PATIENTS:
    for scan_dir in sorted((DATA / patient / "SPECT_Cts").glob("scan*")):
        h = header(scan_dir / "spect")
        rp = h.RadiopharmaceuticalInformationSequence[0]
        t_inj = datetime.strptime(str(rp.RadiopharmaceuticalStartDateTime).split(".")[0], "%Y%m%d%H%M%S")
        t_acq = parse_dt(h.AcquisitionDate, h.AcquisitionTime)
        rows.append({
            "patient": patient,
            "scan": scan_dir.name,
            "t_h": (t_acq - t_inj).total_seconds() / 3600.0,
            "injected_GBq": float(rp.RadionuclideTotalDose) / 1e9,
            "half_life_d": float(rp.RadionuclideHalfLife) / 86400.0,
            "duration_min": float(h.get("ActualFrameDuration", np.nan)) / 60000.0,
            "units": h.get("Units"),
            "decay_correction": h.get("DecayCorrection"),
            "corrections": "/".join(h.get("CorrectedImage", [])),
            "description": h.get("SeriesDescription"),
        })
inventory = pd.DataFrame(rows)
assert set(inventory.decay_correction) <= {"ADMIN", "START"}, inventory.decay_correction.unique()
assert (inventory.groupby("patient").decay_correction.nunique() == 1).all()
inventory

## Loading volumes

For every patient and scan: SPECT (converted to physical activity), CT and masks. The CT is kept on its own grid; it is resampled to the SPECT grid only when saving, so that all files of one scan share the same geometry. The masks are moved to the SPECT grid with nearest-neighbour interpolation if their geometry differs.

In [ ]:
def mask_folders(patient, scan_name):
    voi = DATA / patient / "VOI" / scan_name.capitalize() / "Masks"
    return {f"{group.name.lower()}/{d.name}": d
            for group in sorted(voi.glob("*")) if group.is_dir()
            for d in sorted(group.glob("*")) if d.is_dir()}


data = {}
for patient in PATIENTS:
    data[patient] = {"scans": {}}
    for _, r in inventory[inventory.patient == patient].iterrows():
        scan_dir = DATA / patient / "SPECT_Cts" / r.scan
        raw, _ = read_series(scan_dir / "spect")
        lam = np.log(2) / (r.half_life_d * 86400.0)
        decay = float(np.exp(-lam * r.t_h * 3600.0))
        if r.decay_correction == "ADMIN":
            spect, spect_dc = raw * decay, raw
        else:
            spect, spect_dc = raw, raw / decay
        ct, _ = read_series(scan_dir / "ct")
        masks = {}
        for name, folder in mask_folders(patient, r.scan).items():
            m, _ = read_series(folder)
            m = sitk.Cast(m > 0, sitk.sitkUInt8)
            masks[name] = m if same_grid(m, spect) else to_grid(m, spect, nearest=True)
        data[patient]["scans"][r.scan] = {"t_h": r.t_h, "spect": spect, "spect_dc": spect_dc, "ct": ct, "masks": masks}
    data[patient]["tia"], _ = read_series(DATA / patient / "TIA")
    print(patient, {s: len(v["masks"]) for s, v in data[patient]["scans"].items()},
          "TIA grid matches scan1:", same_grid(data[patient]["tia"], data[patient]["scans"]["scan1"]["spect"]))

## Whole field-of-view activity

Total activity inside the SPECT field of view, physical and decay-corrected to injection, as a fraction of the injected activity. The field of view covers only the abdomen, so the fraction is below 100% even at the first scan. The physical activity must decrease monotonically; the decay-corrected one decreases only because of biological clearance (mostly urinary).

The axial position of the field of view is not the same at every scan (for patient 6 it moves by about 17 cm between scan 1 and scan 4), so the whole-FOV numbers describe slightly different anatomy at each time point and are only qualitative.

In [ ]:
fov = []
for patient in PATIENTS:
    a0 = inventory[inventory.patient == patient].injected_GBq.iloc[0] * 1e9
    for scan, s in data[patient]["scans"].items():
        v = voxel_ml(s["spect"])
        a_phys = float(sitk.GetArrayViewFromImage(s["spect"]).sum()) * v
        a_dc = float(sitk.GetArrayViewFromImage(s["spect_dc"]).sum()) * v
        fov.append({"patient": patient, "scan": scan, "t_h": s["t_h"],
                    "A_phys_MBq": a_phys / 1e6, "A_dc_MBq": a_dc / 1e6,
                    "frac_phys_%": 100 * a_phys / a0, "frac_dc_%": 100 * a_dc / a0})
fov = pd.DataFrame(fov)
fov

## Consistency check of the TIA map

An independent estimate of the TIA from the four physical activities:

- from $0$ to $t_1$: linear uptake from zero to $A(t_1)$;
- between scans: exponential interpolation, $\int_{t_i}^{t_{i+1}} A\,dt = (A_i - A_{i+1})/k_i$ with $k_i = \ln(A_i/A_{i+1})/(t_{i+1}-t_i)$; the trapezoidal rule overestimates fast-decaying curves;
- after the last scan: mono-exponential tail through the last two points, $A(t_n)/k$, with $k$ not smaller than $\lambda_\text{phys}$.

The check is done on the whole field of view and, more importantly, on each organ. The organ activity at each time point comes from the mask drawn on that scan; the MIM TIA is summed inside the scan-1 mask, since the map is registered to scan 1. The organ check is the discriminating one: it excludes the bladder and does not depend on the position of the field of view.

This is not a reference method, so deviations of 10–15% are expected (liver with lesions, registration, early phase). Ratios close to 1 for kidneys and spleen confirm that the MIM map is **physical** TIA in Bq·s/mL and that the decay correction is handled correctly. With the wrong convention the kidney ratios move to about 0.7 (patient 4) or 1.45 (patient 6).

In [ ]:
def tia_estimate(t_s, a_bq, lam_phys):
    total = a_bq[0] * t_s[0] / 2.0
    for i in range(len(t_s) - 1):
        k = np.log(a_bq[i] / a_bq[i + 1]) / (t_s[i + 1] - t_s[i])
        dt = t_s[i + 1] - t_s[i]
        total += (a_bq[i] - a_bq[i + 1]) / k if abs(k * dt) > 1e-9 else a_bq[i] * dt
    k_tail = max(np.log(a_bq[-2] / a_bq[-1]) / (t_s[-1] - t_s[-2]), lam_phys)
    return total + a_bq[-1] / k_tail


def masked_sum(image, mask):
    return float(sitk.GetArrayViewFromImage(image)[sitk.GetArrayViewFromImage(mask).astype(bool)].sum()) * voxel_ml(image)


checks = []
for patient in PATIENTS:
    scans = data[patient]["scans"]
    order = sorted(scans, key=lambda k: scans[k]["t_h"])
    t_s = np.array([scans[k]["t_h"] for k in order]) * 3600.0
    lam_phys = np.log(2) / (inventory[inventory.patient == patient].half_life_d.iloc[0] * 86400.0)
    tia = data[patient]["tia"]

    a_fov = np.array([float(sitk.GetArrayViewFromImage(scans[k]["spect"]).sum()) * voxel_ml(scans[k]["spect"]) for k in order])
    mim_fov = float(sitk.GetArrayViewFromImage(tia).sum()) * voxel_ml(tia)
    checks.append({"patient": patient, "structure": "whole FOV", "TIA_estimate_MBq_h": tia_estimate(t_s, a_fov, lam_phys) / 3.6e9,
                   "TIA_MIM_MBq_h": mim_fov / 3.6e9})

    organs = [n for n in scans["scan1"]["masks"] if n.startswith("organs/")]
    for name in organs:
        a = np.array([masked_sum(scans[k]["spect"], scans[k]["masks"][name]) for k in order])
        mim = masked_sum(tia, scans["scan1"]["masks"][name])
        checks.append({"patient": patient, "structure": name, "TIA_estimate_MBq_h": tia_estimate(t_s, a, lam_phys) / 3.6e9,
                       "TIA_MIM_MBq_h": mim / 3.6e9})

checks = pd.DataFrame(checks)
checks["ratio_MIM_over_estimate"] = checks.TIA_MIM_MBq_h / checks.TIA_estimate_MBq_h
checks.round(3)

## Visual check

Coronal maximum intensity projections of the physical SPECT at each time point and of the TIA map. The colour scale is shared across time points, so the decrease of activity is visible.

In [ ]:
for patient in PATIENTS:
    scans = data[patient]["scans"]
    mips = {k: sitk.GetArrayFromImage(v["spect"]).max(axis=1)[::-1] for k, v in scans.items()}
    vmax = max(m.max() for m in mips.values())
    fig, axes = plt.subplots(1, len(mips) + 1, figsize=(3.2 * (len(mips) + 1), 4))
    for ax, (k, m) in zip(axes, mips.items()):
        ax.imshow(m, cmap="inferno", vmax=vmax / 3, aspect="auto")
        ax.set_title(f"{k}  t = {scans[k]['t_h']:.1f} h")
        ax.axis("off")
    axes[-1].imshow(sitk.GetArrayFromImage(data[patient]["tia"]).max(axis=1)[::-1], cmap="inferno", aspect="auto")
    axes[-1].set_title("TIA map (MIM)")
    axes[-1].axis("off")
    fig.suptitle(f"{patient}: physical activity, coronal MIP")
    fig.tight_layout()

## Preview of the time-activity curves

Total physical activity in each organ and lesion at each time point, from the masks registered to that scan. The full kinetic analysis (fits, model selection, TIA per structure and comparison with the MIM map) is done in notebook 02; here the purpose is only to check that masks and timing make sense.

In [ ]:
tac = []
for patient in PATIENTS:
    for scan, s in data[patient]["scans"].items():
        act = sitk.GetArrayViewFromImage(s["spect"])
        v = voxel_ml(s["spect"])
        for name, m in s["masks"].items():
            mk = sitk.GetArrayViewFromImage(m).astype(bool)
            tac.append({"patient": patient, "scan": scan, "t_h": s["t_h"], "structure": name,
                        "volume_ml": mk.sum() * v, "activity_MBq": act[mk].sum() * v / 1e6})
tac = pd.DataFrame(tac)

fig, axes = plt.subplots(1, len(PATIENTS), figsize=(12, 4), sharey=True)
for ax, patient in zip(axes, PATIENTS):
    for name, g in tac[tac.patient == patient].groupby("structure"):
        g = g.sort_values("t_h")
        ax.plot(g.t_h, g.activity_MBq, "o-", label=name)
    ax.set_yscale("log")
    ax.set_xlabel("Time after injection [h]")
    ax.set_title(patient)
    ax.legend(fontsize=8)
axes[0].set_ylabel("Activity [MBq]")
fig.tight_layout()

tac.pivot_table(index=["patient", "structure"], columns="scan", values="volume_ml").round(1)

The volume table is a quick check of the masks: the volume of each structure should be similar across the four scans, since the contours were drawn on each time point separately.

## Saving

For each patient: one folder per scan with the physical SPECT, the CT resampled to the SPECT grid and the masks, plus the TIA map and a `timing.json` file. All images of one patient share the SPECT grid of the corresponding scan.

In [ ]:
for patient in PATIENTS:
    pdir = OUT / patient
    timing = {}
    for scan, s in data[patient]["scans"].items():
        sdir = pdir / scan
        (sdir / "masks").mkdir(parents=True, exist_ok=True)
        sitk.WriteImage(s["spect"], str(sdir / "spect_bqml.nii.gz"))
        sitk.WriteImage(to_grid(s["ct"], s["spect"], default=-1000.0), str(sdir / "ct_on_spect.nii.gz"))
        for name, m in s["masks"].items():
            sitk.WriteImage(m, str(sdir / "masks" / f"{name.replace('/', '_')}.nii.gz"))
        timing[scan] = s["t_h"]
    sitk.WriteImage(data[patient]["tia"], str(pdir / "tia_mim_bqml_s.nii.gz"))
    meta = inventory[inventory.patient == patient].iloc[0]
    (pdir / "timing.json").write_text(json.dumps({
        "t_h": timing,
        "injected_Bq": float(meta.injected_GBq) * 1e9,
        "half_life_s": float(meta.half_life_d) * 86400.0,
        "source_decay_correction": meta.decay_correction,
        "spect_units": "Bq/mL, physical activity at acquisition start",
    }, indent=2))
    print("saved", pdir)